# Fine-tune YOLO11 (từ `yolo11s_tci.pt`) cho phát hiện tàu trên bộ dữ liệu của bạn

**Vì sao cần fine-tune.** Checkpoint `yolo11s_tci.pt` của bài báo Mäyrä et al. (2025) được huấn luyện trên ảnh
**bờ biển Phần Lan, Sentinel-2 L1C-TCI**. Bộ dữ liệu của bạn (tile **51RUM — Đông Á**, mức **L2A**, màu khác) là
một **miền khác hẳn**, nên dùng thẳng model (zero-shot) cho kết quả rất thấp (F1 ≈ 0.17). Cách đúng là
**transfer learning**: lấy `yolo11s_tci.pt` làm trọng số khởi tạo rồi **fine-tune trên tập train của bạn**, sau đó
đánh giá trên tập test.

**Notebook này:**
1. Tải `yolo11s_tci.pt` (HuggingFace `mayrajeo/marine-vessel-yolo`) làm trọng số khởi tạo.
2. Chuẩn bị train/valid/test (định dạng YOLOv11: `images/` + `labels/`).
3. **Fine-tune** YOLO11 (`single_cls`, cấu hình theo repo [mayrajeo/ship-detection](https://github.com/mayrajeo/ship-detection)).
4. Đánh giá **best.pt** trên test: Precision/Recall/F1/mAP50/mAP50-95 + TP/FP/FN.
5. **So sánh** baseline (zero-shot) vs sau fine-tune, và **visualise** ảnh test.

**Môi trường:** chỉ cần `ultralytics` (Kaggle GPU **T4** — tránh P100 vì lỗi CUDA kernel). Độc lập với InternImage/mmdet.

## Section 0 — Cấu hình (SỬA TRƯỚC KHI CHẠY)

Bộ dữ liệu đã ở định dạng YOLOv11: mỗi split có `images/` + `labels/`. Trỏ đúng đường dẫn train/valid/test.
Bật **Settings → Accelerator → GPU T4 x2**.

In [ ]:
import os

CFG = dict(
    # ===== Trọng số khởi tạo (fine-tune từ đây) =====
    HF_REPO_ID    = "mayrajeo/marine-vessel-yolo",
    HF_FILENAME   = "yolo11s_tci.pt",
    LOCAL_WEIGHTS = "",   # nếu đã add .pt làm Kaggle Dataset, trỏ đường dẫn để khỏi tải từ HF

    # ===== Dữ liệu (định dạng YOLO: images/ + labels/ cho mỗi split) =====
    TRAIN_IMAGES_DIR = "/kaggle/input/datasets/hunglq/annotated-sentinel/train/images",
    TRAIN_LABELS_DIR = "/kaggle/input/datasets/hunglq/annotated-sentinel/train/labels",
    VAL_IMAGES_DIR   = "/kaggle/input/datasets/hunglq/annotated-sentinel/valid/images",
    VAL_LABELS_DIR   = "/kaggle/input/datasets/hunglq/annotated-sentinel/valid/labels",
    TEST_IMAGES_DIR  = "/kaggle/input/datasets/hunglq/annotated-sentinel/test/images",
    TEST_LABELS_DIR  = "/kaggle/input/datasets/hunglq/annotated-sentinel/test/labels",
    CLASS_NAME       = "vessel",

    # ===== Siêu tham số fine-tune (theo repo mayrajeo: single_cls, scale=0.25, flipud=0.5, optimizer auto) =====
    EPOCHS     = 100,     # giảm nếu Kaggle hết giờ; đã bật early-stopping bằng PATIENCE
    PATIENCE   = 20,      # dừng sớm nếu val không cải thiện sau ngần này epoch
    IMG_SIZE   = 800,     # ảnh 800x800; quét imgsz cho thấy 800 tốt nhất với bộ này
    BATCH      = -1,      # -1 = tự chọn theo ~60% VRAM (giống bài báo). Đặt số cố định (vd 16) nếu muốn.
    OPTIMIZER  = "auto",  # auto -> AdamW, lr0=0.001
    LR0        = 0.001,
    SCALE      = 0.25,    # augmentation (mayrajeo)
    FLIPUD     = 0.5,
    FLIPLR     = 0.5,
    FREEZE     = 0,       # 0 = fine-tune toàn bộ; đặt 10 để đóng băng backbone nếu dữ liệu quá ít / overfit

    # ===== Đánh giá / visualise =====
    CONF_EVAL  = 0.001,
    CONF_VIS   = 0.25,
    IOU_NMS    = 0.7,
    N_VIS      = 12,

    DEVICE     = "auto",  # "auto" | "0" | "cpu"
    SEED       = 42,

    # ===== Thư mục làm việc =====
    WORK_ROOT  = "/kaggle/working",
    DATA_DIR   = "/kaggle/working/yolo_ds",          # nơi copy train/val/test + data.yaml
    OUT_DIR    = "/kaggle/working/finetune_outputs", # nơi lưu model + metric + hình
)

os.makedirs(CFG["DATA_DIR"], exist_ok=True)
os.makedirs(CFG["OUT_DIR"], exist_ok=True)

for k in ("TRAIN_IMAGES_DIR", "VAL_IMAGES_DIR", "TEST_IMAGES_DIR"):
    assert os.path.isdir(CFG[k]), f"Không thấy thư mục ảnh: {CFG[k]}"
print("CFG OK.")

## Section 1 — GPU & môi trường

> ⚠️ Lỗi `CUDA error: no kernel image is available...` = torch không hỗ trợ kiến trúc GPU (hay gặp P100).
> Đổi **Accelerator → GPU T4 x2** + **Restart** rồi chạy lại. Notebook cũng tự rơi về CPU (Section 2) nếu cần,
> nhưng **fine-tune trên CPU cực chậm** — hãy dùng GPU.

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## Section 2 — Cài `ultralytics` + chọn thiết bị

In [ ]:
%%capture install_log
!pip install -q -U ultralytics "huggingface_hub>=0.24"

In [ ]:
import ultralytics
ultralytics.checks()
print("ultralytics:", ultralytics.__version__)

In [ ]:
import torch

def pick_device(pref):
    if pref not in ("auto", None, ""):
        print("DEVICE ép:", pref); return str(pref)
    if not torch.cuda.is_available():
        print("CUDA không khả dụng -> CPU"); return "cpu"
    try:
        _ = (torch.zeros(16, device="cuda") + 1).sum().item(); torch.cuda.synchronize()
        print("GPU OK:", torch.cuda.get_device_name(0)); return "0"
    except Exception as e:
        print("⚠️ GPU không chạy kernel:", str(e).splitlines()[0], "-> CPU (đổi sang T4 để train được!)")
        return "cpu"

DEVICE_EFF = pick_device(CFG["DEVICE"])
print("DEVICE_EFF =", DEVICE_EFF)
if DEVICE_EFF == "cpu":
    print("⚠️ CẢNH BÁO: sẽ fine-tune trên CPU, rất chậm. Nên đổi Accelerator sang GPU T4 x2.")

## Section 3 — Tải trọng số khởi tạo `yolo11s_tci.pt`

In [ ]:
def resolve_weights(cfg):
    if cfg["LOCAL_WEIGHTS"] and os.path.exists(cfg["LOCAL_WEIGHTS"]):
        print("Dùng weights local:", cfg["LOCAL_WEIGHTS"]); return cfg["LOCAL_WEIGHTS"]
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(repo_id=cfg["HF_REPO_ID"], filename=cfg["HF_FILENAME"],
                        local_dir=os.path.join(cfg["WORK_ROOT"], "weights"))
    print("Đã tải:", p); return p

INIT_WEIGHTS = resolve_weights(CFG)

## Section 4 — Chuẩn bị dữ liệu YOLO (train / val / test)

Copy ảnh + labels của từng split vào `DATA_DIR/{split}/images|labels` (để ultralytics ghi được `labels.cache`,
vì `/kaggle/input` chỉ đọc), ép mọi class về 0 (single-cls), rồi ghi `data.yaml`.

In [ ]:
import shutil, glob
from pathlib import Path

IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

def _reset(d):
    d = Path(d)
    if d.exists(): shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d

def build_split(split, img_dir, lbl_dir, cfg):
    out_img = _reset(Path(cfg["DATA_DIR"]) / split / "images")
    out_lbl = _reset(Path(cfg["DATA_DIR"]) / split / "labels")
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir) if lbl_dir else img_dir.parent / "labels"
    imgs = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    n_img, n_box = 0, 0
    for src in imgs:
        shutil.copy(src, out_img / src.name)
        sl = lbl_dir / (src.stem + ".txt")
        fixed = []
        if sl.exists():
            for ln in sl.read_text().splitlines():
                p = ln.split()
                if len(p) >= 5:
                    fixed.append("0 " + " ".join(p[1:5])); n_box += 1
        (out_lbl / (src.stem + ".txt")).write_text("\n".join(fixed))
        n_img += 1
    print(f"  [{split:5s}] {n_img} ảnh, {n_box} box")
    return n_img, n_box

print("Chuẩn bị dữ liệu:")
n_tr = build_split("train", CFG["TRAIN_IMAGES_DIR"], CFG["TRAIN_LABELS_DIR"], CFG)
n_va = build_split("val",   CFG["VAL_IMAGES_DIR"],   CFG["VAL_LABELS_DIR"],   CFG)
n_te = build_split("test",  CFG["TEST_IMAGES_DIR"],  CFG["TEST_LABELS_DIR"],  CFG)
assert n_tr[0] > 0 and n_te[0] > 0

In [ ]:
import yaml
DATA_YAML = os.path.join(CFG["DATA_DIR"], "data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump({
        "path": CFG["DATA_DIR"],
        "train": "train/images",
        "val":   "val/images",
        "test":  "test/images",
        "names": {0: CFG["CLASS_NAME"]},
    }, f, sort_keys=False, allow_unicode=True)
print(open(DATA_YAML).read())

## Section 5 — (Tuỳ chọn) Baseline zero-shot trước khi fine-tune

Đánh giá thẳng `yolo11s_tci.pt` trên test để có mốc so sánh "trước fine-tune". Dự kiến rất thấp do domain gap.

In [ ]:
from ultralytics import YOLO

def eval_on_test(weights, tag):
    m = YOLO(weights)
    mt = m.val(data=DATA_YAML, split="test", imgsz=CFG["IMG_SIZE"], conf=CFG["CONF_EVAL"],
               iou=CFG["IOU_NMS"], single_cls=True, device=DEVICE_EFF, plots=False,
               project=CFG["OUT_DIR"], name=f"eval_{tag}", exist_ok=True, verbose=False)
    P, R = float(mt.box.mp), float(mt.box.mr)
    return {"model": tag, "P": round(P, 3), "R": round(R, 3),
            "F1": round(2 * P * R / (P + R + 1e-9), 3),
            "mAP50": round(float(mt.box.map50), 3), "mAP50-95": round(float(mt.box.map), 3)}

baseline = eval_on_test(INIT_WEIGHTS, "baseline_zeroshot")
print("Baseline (zero-shot):", baseline)

## Section 6 — Fine-tune YOLO11

Khởi tạo từ `yolo11s_tci.pt` và huấn luyện trên tập train của bạn. Ultralytics tự lưu `best.pt` (val tốt nhất) và
`last.pt`, cùng các biểu đồ (loss, PR, F1...) trong `OUT_DIR/finetune/`.

> ⏱️ Tuỳ số ảnh và epoch, bước này có thể mất từ vài chục phút đến vài giờ trên T4. Nếu sắp hết 12h của Kaggle,
> giảm `EPOCHS` hoặc dựa vào `PATIENCE` để dừng sớm.

In [ ]:
batch = CFG["BATCH"] if DEVICE_EFF != "cpu" else 4   # CPU: batch nhỏ cho đỡ kẹt

model = YOLO(INIT_WEIGHTS)
train_res = model.train(
    data=DATA_YAML,
    epochs=CFG["EPOCHS"],
    patience=CFG["PATIENCE"],
    imgsz=CFG["IMG_SIZE"],
    batch=batch,
    optimizer=CFG["OPTIMIZER"],
    lr0=CFG["LR0"],
    scale=CFG["SCALE"],
    flipud=CFG["FLIPUD"],
    fliplr=CFG["FLIPLR"],
    single_cls=True,
    freeze=CFG["FREEZE"],
    device=DEVICE_EFF,
    seed=CFG["SEED"],
    project=CFG["OUT_DIR"],
    name="finetune",
    exist_ok=True,
    plots=True,
)
BEST = os.path.join(CFG["OUT_DIR"], "finetune", "weights", "best.pt")
print("best.pt:", BEST, "| tồn tại:", os.path.exists(BEST))

## Section 7 — Đánh giá `best.pt` trên test + so sánh trước/sau

In [ ]:
from ultralytics import YOLO
import numpy as np, json as _json

best_model = YOLO(BEST)
metrics = best_model.val(
    data=DATA_YAML, split="test", imgsz=CFG["IMG_SIZE"], conf=CFG["CONF_EVAL"],
    iou=CFG["IOU_NMS"], single_cls=True, device=DEVICE_EFF, plots=True,
    project=CFG["OUT_DIR"], name="test_eval", exist_ok=True, verbose=True,
)

P, R = float(metrics.box.mp), float(metrics.box.mr)
F1 = 2 * P * R / (P + R + 1e-9)
cm = metrics.confusion_matrix.matrix
TP, FP, FN = int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0])
results = {
    "precision": round(P, 4), "recall": round(R, 4), "f1": round(F1, 4),
    "mAP50": round(float(metrics.box.map50), 4), "mAP50_95": round(float(metrics.box.map), 4),
    "TP": TP, "FP": FP, "FN": FN,
    "num_test_images": int(n_te[0]), "num_gt_boxes": int(n_te[1]),
}
with open(os.path.join(CFG["OUT_DIR"], "test_metrics.json"), "w") as f:
    _json.dump(results, f, indent=2)

print("======= KẾT QUẢ TEST (sau fine-tune) =======")
for k, v in results.items():
    print(f"  {k:16s}: {v}")

In [ ]:
import pandas as pd
from IPython.display import display

finetuned = {"model": "finetuned_best", "P": results["precision"], "R": results["recall"],
             "F1": results["f1"], "mAP50": results["mAP50"], "mAP50-95": results["mAP50_95"]}
cmp_df = pd.DataFrame([baseline, finetuned])
print("So sánh trước/sau fine-tune trên test:")
display(cmp_df)
cmp_df.to_csv(os.path.join(CFG["OUT_DIR"], "compare_baseline_vs_finetune.csv"), index=False)

# Hiển thị biểu đồ ultralytics
import matplotlib.pyplot as plt, matplotlib.image as mpimg
tdir = os.path.join(CFG["OUT_DIR"], "test_eval")
for name in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    fp = os.path.join(tdir, name)
    if os.path.exists(fp):
        plt.figure(figsize=(6, 5)); plt.imshow(mpimg.imread(fp)); plt.axis("off"); plt.title(name); plt.show()

### Đường cong huấn luyện (results.png)

In [ ]:
import os, matplotlib.pyplot as plt, matplotlib.image as mpimg
rp = os.path.join(CFG["OUT_DIR"], "finetune", "results.png")
if os.path.exists(rp):
    plt.figure(figsize=(12, 8)); plt.imshow(mpimg.imread(rp)); plt.axis("off"); plt.show()

## Section 8 — Visualise dự đoán trên test (GT xanh vs dự đoán đỏ)

In [ ]:
import random, glob
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from PIL import Image

def read_yolo_gt(label_path, W, H):
    boxes = []
    if not os.path.exists(label_path): return boxes
    for ln in open(label_path):
        p = ln.split()
        if len(p) < 5: continue
        xc, yc, w, h = map(float, p[1:5])
        boxes.append(((xc - w / 2) * W, (yc - h / 2) * H, w * W, h * H))
    return boxes

test_img_dir = os.path.join(CFG["DATA_DIR"], "test", "images")
test_lbl_dir = os.path.join(CFG["DATA_DIR"], "test", "labels")
img_paths = sorted(glob.glob(os.path.join(test_img_dir, "*")))
random.seed(CFG["SEED"]); random.shuffle(img_paths)
sel = img_paths[:CFG["N_VIS"]]
preds = best_model.predict(sel, imgsz=CFG["IMG_SIZE"], conf=CFG["CONF_VIS"],
                           iou=CFG["IOU_NMS"], device=DEVICE_EFF, verbose=False)

ncol = 3; nrow = (len(sel) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(6 * ncol, 6 * nrow))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
vis_dir = os.path.join(CFG["OUT_DIR"], "visualisations"); os.makedirs(vis_dir, exist_ok=True)

for ax, img_path, res in zip(axes, sel, preds):
    im = Image.open(img_path).convert("RGB"); W, H = im.size
    ax.imshow(im)
    for (x, y, w, h) in read_yolo_gt(os.path.join(test_lbl_dir, Path(img_path).stem + ".txt"), W, H):
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="lime", linewidth=1.5))
    n_pred = 0
    if res.boxes is not None and len(res.boxes) > 0:
        xyxy = res.boxes.xyxy.cpu().numpy(); confs = res.boxes.conf.cpu().numpy(); n_pred = len(xyxy)
        for (x1, y1, x2, y2), c in zip(xyxy, confs):
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="red", linewidth=1.2))
            ax.text(x1, max(y1 - 2, 0), f"{c:.2f}", color="red", fontsize=7,
                    bbox=dict(facecolor="white", alpha=0.6, pad=0, edgecolor="none"))
    gts = read_yolo_gt(os.path.join(test_lbl_dir, Path(img_path).stem + ".txt"), W, H)
    ax.set_title(f"{Path(img_path).name}\nGT={len(gts)}  Pred={n_pred}", fontsize=9); ax.axis("off")
for ax in axes[len(sel):]: ax.axis("off")
fig.legend(handles=[Line2D([0], [0], color="lime", lw=2, label="Ground truth"),
                    Line2D([0], [0], color="red", lw=2, label=f"Prediction (conf≥{CFG['CONF_VIS']})")],
           loc="upper center", ncol=2, fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.98])
out_fig = os.path.join(vis_dir, "test_predictions_grid.png")
plt.savefig(out_fig, dpi=120, bbox_inches="tight"); print("Đã lưu:", out_fig); plt.show()

## Section 9 — Ghi chú

- **Đây mới là cách so sánh công bằng với dữ liệu của bạn:** fine-tune thích nghi model với ảnh Đông Á / L2A của bạn,
  thay vì kỳ vọng model Phần Lan chạy tốt zero-shot.
- **Nếu vẫn thấp sau fine-tune:** kiểm tra lại chất lượng nhãn (Section 8 — box xanh có ôm đúng tàu không), tăng
  `EPOCHS`, hoặc thử `FREEZE=10` (đóng băng backbone) nếu train ít dữ liệu bị overfit.
- **Tăng recall cho tàu nhỏ:** có thể thử `IMG_SIZE=1024/1280` khi train *và* eval (giữ nhất quán hai bên).
- **Trọng số kết quả:** `finetune_outputs/finetune/weights/best.pt` — dùng cho suy luận về sau (hoặc SAHI trên
  ảnh lớn như bài báo).
- **Lưu lại:** mọi thứ nằm trong `/kaggle/working/finetune_outputs/` (model, metric, biểu đồ, ảnh visualise).
- **Kaggle giới hạn 12h/phiên:** nếu train dài, bật *Save Version → Run All* để chạy nền, hoặc giảm epoch.